# 🔄 使用 Azure OpenAI（Responses API）的基本代理工作流程 (.NET)

## 📋 工作流程編排教學

本筆記本示範如何使用 Microsoft Agent Framework for .NET 及 Azure OpenAI（Responses API）構建複雜的 <strong>代理工作流程</strong>。您將學習創建多步驟的業務流程，在這些流程中 AI 代理通過結構化的編排模式協作完成複雜任務。

## 🎯 學習目標

### 🏗️ <strong>工作流程架構基礎</strong>
- <strong>工作流程建構器</strong>：設計和編排複雜的多步驟 AI 流程
- <strong>代理協調</strong>：協調工作流程中多個專門化代理
- **Azure OpenAI（Responses API）**：在工作流程中利用 Azure OpenAI Responses API
- <strong>可視化工作流程設計</strong>：創建並可視化工作流程結構以加深理解

### 🔄 <strong>流程編排模式</strong>
- <strong>順序處理</strong>：以邏輯順序串連多個代理任務
- <strong>狀態管理</strong>：在工作流程階段間維持上下文與資料流
- <strong>錯誤處理</strong>：實現強韌的錯誤恢復及工作流程彈性
- <strong>性能優化</strong>：為企業級運營設計高效工作流程

### 🏢 <strong>企業工作流程應用</strong>
- <strong>業務流程自動化</strong>：自動化複雜的組織性工作流程
- <strong>內容生產管線</strong>：具審核及批准階段的編輯工作流程
- <strong>客戶服務自動化</strong>：多步驟客戶查詢解決流程
- <strong>資料處理工作流程</strong>：具有 AI 支援轉換的 ETL 工作流程

## ⚙️ 前置條件與設定

### 📦 **必要 NuGet 套件**

本工作流程示範使用數個關鍵 .NET 套件：

```xml
<!-- Core AI Framework -->
<PackageReference Include="Microsoft.Extensions.AI" Version="9.9.0" />

<!-- Azure OpenAI (Responses API) -->
<PackageReference Include="Azure.AI.OpenAI" Version="2.1.0" />
<PackageReference Include="Azure.Identity" Version="1.13.1" />

<!-- Agent Framework (Local Development) -->
<!-- Microsoft.Agents.AI.dll - Core agent abstractions -->
<!-- Microsoft.Agents.AI.OpenAI.dll - Azure OpenAI (Responses API) integration -->

<!-- Configuration and Environment -->
<PackageReference Include="DotNetEnv" Version="3.1.1" />
```

### 🔑 **Azure OpenAI 配置**

**環境設定 (.env 檔案)：**
```env
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com
AZURE_OPENAI_DEPLOYMENT=gpt-4.1-mini
```

**Azure OpenAI 存取：**
1. 在 Azure 入口網站建立 Azure OpenAI 資源
2. 部署模型（例如 `gpt-4.1-mini`）並記下部署名稱
3. 使用 `az login` 登入，並依上述設定環境變數

### 🏗️ <strong>工作流程架構概述</strong>

```mermaid
graph TD
    A[工作流程構建器] --> B[代理註冊中心]
    B --> C[工作流程執行引擎]
    C --> D[Agent 1: 內容生成器]
    C --> E[Agent 2: 內容審核員] 
    D --> F[工作流程結果]
    E --> F
    G[Azure OpenAI（回應 API）] --> D
    G --> E
```

**主要組件：**
- **WorkflowBuilder**：設計工作流程的主要編排引擎
- **AIAgent**：具有特定能力的個別專門代理
- **Azure OpenAI 客戶端**：Azure OpenAI Responses API 集成
- <strong>執行上下文</strong>：管理工作流程階段間的狀態與資料流

## 🎨 <strong>企業工作流程設計模式</strong>

### 📝 <strong>內容生產工作流程</strong>
```
User Request → Content Generation → Quality Review → Final Output
```

### 🔍 <strong>文件處理管線</strong>
```
Document Input → Analysis → Extraction → Validation → Structured Output
```

### 💼 <strong>商業智慧工作流程</strong>
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

### 🤝 <strong>客戶服務自動化</strong>
```
Customer Inquiry → Classification → Processing → Response Generation → Follow-up
```

## 🏢 <strong>企業效益</strong>

### 🎯 <strong>可靠性與可擴展性</strong>
- <strong>確定性執行</strong>：一致且可重複的工作流程結果
- <strong>錯誤恢復</strong>：在工作流程任一階段優雅地處理失敗
- <strong>性能監控</strong>：追蹤執行指標及優化機會
- <strong>資源管理</strong>：高效配置及利用 AI 模型資源

### 🔒 <strong>安全性與合規</strong>
- <strong>安全驗證</strong>：透過 `az login` （AzureCliCredential）使用 Microsoft Entra ID 驗證
- <strong>稽核追蹤</strong>：完整紀錄工作流程執行與決策點
- <strong>存取控制</strong>：工作流程執行及監控的細粒度權限
- <strong>資料隱私</strong>：工作流程中敏感資訊的安全處理

### 📊 <strong>可觀察性與管理</strong>
- <strong>可視化工作流程設計</strong>：清晰呈現流程走向與依賴關係
- <strong>執行監控</strong>：即時追蹤工作流程進度與效能
- <strong>錯誤報告</strong>：詳細錯誤分析與除錯功能
- <strong>性能分析</strong>：優化和容量規劃的指標

讓我們一起打造您的首個企業級 AI 工作流程吧！🚀


In [1]:
#r "nuget: Microsoft.Extensions.AI, 9.9.1"

Installed Packages Microsoft.Extensions.AI, 9.9.1

In [ ]:
#r "nuget: Azure.AI.OpenAI, 2.1.0"

Installed Packages System.ClientModel, 1.6.1

In [3]:
#r "nuget: Azure.Identity, 1.15.0"
#r "nuget: System.Linq.Async, 6.0.3"
#r "nuget: OpenTelemetry.Api, 1.0.0"
#r "nuget: OpenTelemetry.Api, 1.0.0"

Installed Packages Azure.Identity, 1.15.0 OpenTelemetry.Api, 1.0.1 System.Linq.Async, 6.0.3

In [5]:

#r "nuget: Microsoft.Agents.AI.Workflows, 1.0.0-preview.251001.3"

Installed Packages Microsoft.Agents.AI.Workflows, 1.0.0-preview.251001.3

In [ ]:

#r "nuget: Microsoft.Agents.AI.OpenAI, 1.0.0-preview.251001.3"

Installed Packages Microsoft.Agents.AI.OpenAI, 1.0.0-preview.251001.2

In [7]:
#r "nuget: DotNetEnv, 3.1.1"

Installed Packages DotNetEnv, 3.1.1

In [8]:
// #r "nuget: Microsoft.Extensions.AI.OpenAI, 9.9.0-preview.1.25458.4"

In [ ]:
using System;
using System.ComponentModel;
using Azure.AI.OpenAI;
using Azure.Identity;
using Microsoft.Extensions.AI;
using Microsoft.Agents.AI;
using Microsoft.Agents.AI.Workflows;

In [10]:
 using DotNetEnv;

In [11]:
Env.Load("../../../.env");

In [ ]:
// Azure OpenAI with the Responses API (stable v1 endpoint). Sign in with `az login`.
var azureEndpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT") ?? throw new InvalidOperationException("AZURE_OPENAI_ENDPOINT is not set.");
var deployment = Environment.GetEnvironmentVariable("AZURE_OPENAI_DEPLOYMENT") ?? "gpt-4.1-mini";

In [ ]:
// The Azure OpenAI client is created directly from the endpoint and Azure CLI credential — no custom client options are required.

In [ ]:
var azureClient = new AzureOpenAIClient(new Uri(azureEndpoint), new AzureCliCredential());

In [15]:
const string ReviewerAgentName = "Concierge";
const string ReviewerAgentInstructions = @"
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. ";

In [16]:
const string FrontDeskAgentName = "FrontDesk";
const string FrontDeskAgentInstructions = @"""
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """;

In [ ]:
AIAgent reviewerAgent = azureClient.GetOpenAIResponseClient(deployment).CreateAIAgent(
    name:ReviewerAgentName,instructions:ReviewerAgentInstructions);
AIAgent frontDeskAgent  = azureClient.GetOpenAIResponseClient(deployment).CreateAIAgent(
    name:FrontDeskAgentName,instructions:FrontDeskAgentInstructions);

In [18]:
var workflow = new WorkflowBuilder(frontDeskAgent)
            .AddEdge(frontDeskAgent, reviewerAgent)
            .Build();

In [19]:
ChatMessage userMessage = new ChatMessage(ChatRole.User, [
	new TextContent("I would like to go to Paris.") 
]);

In [20]:
StreamingRun run = await InProcessExecution.StreamAsync(workflow, userMessage);

In [21]:
await run.TrySendMessageAsync(new TurnToken(emitEvents: true));
string id="";
string messageData="";
await foreach (WorkflowEvent evt in run.WatchStreamAsync().ConfigureAwait(false))
{
    if (evt is AgentRunUpdateEvent executorComplete)
    {
        if(id=="")
        {
            id=executorComplete.ExecutorId;
        }
        if(id==executorComplete.ExecutorId)
        {
            messageData+=executorComplete.Data.ToString();
        }
        else
        {
            id=executorComplete.ExecutorId;
        }
        // Console.WriteLine($"{executorComplete.ExecutorId}: {executorComplete.Data}");
    }
}

Console.WriteLine(messageData);

Visit the Louvre Museum. It's a must-see for art enthusiasts and history lovers.That recommendation is quite popular and likely to attract many tourists. To refine it for a more local and authentic experience, consider suggesting an alternative that focuses on smaller, lesser-known art venues or galleries. Look for places where local artists exhibit or community spaces that host cultural events. This approach allows travelers to connect with the local art scene more intimately, away from the typical tourist routes.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
